# Faruq-v3 — AF2 Direct From Official YOLO26n Pretrained — Kaggle

Seed-42 paired promotion screen:

`official yolo26n.pt -> D0DIRECT` versus `official yolo26n.pt -> AF2DIRECT`.

No D0/D0FT/coffee checkpoint is used as an intermediate parent. Both arms use the same 50-epoch maximum schedule and matched 21-class target-head initialization. Validation only; test stays locked.

Required Kaggle input: `faruq-development-v3-grouped.tar.bin`.

GPU + Internet ON. The notebook fails closed before training if the direct static preflight does not pass.


In [ ]:
from pathlib import Path
import csv, hashlib, importlib, json, os, shutil, subprocess, sys, time, zipfile

INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir():
    raise RuntimeError('Kaggle-only notebook')
os.chdir(WORK)

print('INDEXING /kaggle/input ONCE ...',flush=True)
t0=time.time(); INPUT_INDEX={}; file_count=0
for root,dirs,files in os.walk(INPUT):
    for filename in files:
        p=Path(root)/filename
        INPUT_INDEX.setdefault(filename,[]).append(p)
        file_count+=1
print(f'INPUT INDEX READY: {file_count} files in {time.time()-t0:.2f}s',flush=True)

def input_matches(name):
    return sorted(INPUT_INDEX.get(name,[]))

def input_one(name):
    matches=input_matches(name)
    if len(matches)!=1:
        raise FileNotFoundError(f'Harus tepat satu {name}; ditemukan {matches}')
    return matches[0]

ARCHIVE=input_one('faruq-development-v3-grouped.tar.bin')
print('DATA INPUT OK:',ARCHIVE)


In [ ]:
import torch

BRANCH='codex/af2-direct-from-pretrained'
REPO=WORK/'coffee-bean-detection'
os.chdir(WORK)
if REPO.exists():
    shutil.rmtree(REPO)

for attempt in range(1,4):
    r=subprocess.run(
        ['git','clone','--depth','1','--branch',BRANCH,
         'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],
        cwd=WORK,
    )
    if r.returncode==0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt==3:
        raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)

subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)

for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'):
        sys.modules.pop(name,None)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)

COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('BRANCH:',BRANCH)
print('COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All')

preflight=subprocess.run(
    [sys.executable,'-m','pytest','-q','tests/test_afab.py','tests/test_af2_direct_protocol.py'],
    cwd=REPO,text=True,capture_output=True,
)
print(preflight.stdout,flush=True)
if preflight.returncode:
    print(preflight.stderr,flush=True)
    raise RuntimeError(f'DIRECT CONTRACT TESTS FAILED rc={preflight.returncode}')
print('ALL DIRECT PRETRAIN CONTRACT TESTS PASS')


In [ ]:
from coffee_detector.experiments.prepare_faruq_v3_kaggle import prepare_faruq_v3_kaggle_input

DATA,CORE=prepare_faruq_v3_kaggle_input(INPUT,WORK)
if CORE.get('decision')!='PASS' or CORE.get('test_images_accessed') is not False:
    raise RuntimeError('Dataset contract gagal')
if (DATA/'test').exists():
    raise RuntimeError('TEST TEREXPOSE — STOP')
GROUPED=DATA/'faruq_grouped_summary.json'

# Resolve the official pretrained file once. Both arms receive this exact path/SHA.
os.chdir(REPO)
from ultralytics import YOLO
_ = YOLO('yolo26n.pt')
PRETRAINED=(REPO/'yolo26n.pt').resolve()
if not PRETRAINED.is_file():
    raise FileNotFoundError('Ultralytics tidak menghasilkan yolo26n.pt lokal')

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

print('OFFICIAL PRETRAINED:',PRETRAINED)
print('PRETRAINED SHA256:',sha256(PRETRAINED))

OUT=WORK/'af2-direct-from-pretrained-seed42-v1'
STATE_ZIP=WORK/'af2-direct-from-pretrained-seed42-state.zip'

resume_states=input_matches(STATE_ZIP.name)
if len(resume_states)>1:
    raise RuntimeError(f'Resume state ambigu: {resume_states}')
if len(resume_states)==1 and not OUT.exists():
    print('RESTORE STATE:',resume_states[0],flush=True)
    with zipfile.ZipFile(resume_states[0],'r') as z:
        z.extractall(WORK)
OUT.mkdir(parents=True,exist_ok=True)
print('OUTPUT:',OUT)


In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2_direct import run_direct_static_preflight

STATIC=OUT/'direct_static_preflight.json'
if STATIC.is_file():
    cached=json.loads(STATIC.read_text(encoding='utf-8'))
    if (
        cached.get('decision')!='PASS'
        or cached.get('training_authorized') is not True
        or cached.get('pretrained_checkpoint_sha256')!=sha256(PRETRAINED)
        or cached.get('test_images_accessed') is not False
    ):
        raise RuntimeError('Cached static preflight stale/invalid')
    static=cached
    print('REUSE STATIC PREFLIGHT')
else:
    static=run_direct_static_preflight(PRETRAINED,STATIC,seed=42)

failed=[k for k,v in static['gates'].items() if not v]
if failed:
    raise RuntimeError(f'STATIC PREFLIGHT FAIL: {failed}')
print('STATIC PREFLIGHT PASS')
print('COMMON INIT SHA:',static['common_initialized_detector_state_sha256'])
print('PARAMS native/candidate:',static['native_parameter_count'],static['candidate_parameter_count'])
print('AF2 LEARNED PARAMS:',static['af2_learned_parameter_count'])
print('AF2 PROBE MAX ABS CHANGE:',static['af2_probe_max_abs_change'])


In [ ]:
def epoch_count(arm):
    path=OUT/arm/f'{arm}_seed42'/'results.csv'
    if not path.is_file():
        return 0
    try:
        with path.open(newline='',encoding='utf-8') as f:
            return sum(1 for _ in csv.DictReader(f))
    except Exception:
        return 0

def snapshot_state():
    if STATE_ZIP.exists():
        STATE_ZIP.unlink()
    archive=Path(shutil.make_archive(str(STATE_ZIP.with_suffix('')),'zip',root_dir=WORK,base_dir=OUT.name))
    print('STATE SNAPSHOT:',archive,archive.stat().st_size,'bytes',flush=True)
    return archive

LOG=OUT/'af2_direct_run.log'
cmd=[
    sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_direct',
    '--data-root',str(DATA),
    '--grouped-summary',str(GROUPED),
    '--pretrained-checkpoint',str(PRETRAINED),
    '--output-root',str(OUT),
    '--seed','42',
    '--device','0',
    '--authorize-training',
]
print('START/RESUME PAIRED DIRECT SCREEN | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)

shown=None
snap_control=False
while process.poll() is None:
    status=(epoch_count('D0DIRECT'),epoch_count('AF2DIRECT'))
    if status!=shown:
        print(f'D0DIRECT: {status[0]}/50 epoch tercatat | AF2DIRECT: {status[1]}/50',flush=True)
        shown=status
    control_result=OUT/'val_reports'/'D0DIRECT_seed42_result.json'
    if control_result.is_file() and not snap_control:
        payload=json.loads(control_result.read_text(encoding='utf-8'))
        print('CONTROL DONE:',payload['metrics'],payload['diagnostic'],flush=True)
        snapshot_state()
        snap_control=True
    time.sleep(120)

if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-200:]))
    snapshot_state()
    raise RuntimeError(f'DIRECT RUN FAILED rc={process.returncode}')

snapshot_state()
SUMMARY=OUT/'af2_direct_seed42_summary.json'
if not SUMMARY.is_file():
    raise FileNotFoundError(SUMMARY)
print('PAIRED DIRECT SCREEN COMPLETE')


In [ ]:
from IPython.display import display
import pandas as pd

summary=json.loads(SUMMARY.read_text(encoding='utf-8'))
control=summary['control']; candidate=summary['candidate']
delta=summary['deltas_af2direct_minus_d0direct']

rows=[
    {'metric':'Macro mAP50-95','D0DIRECT':control['metrics']['macro_map50_95'],'AF2DIRECT':candidate['metrics']['macro_map50_95'],'delta':delta['macro_map50_95']},
    {'metric':'Bottom-3 mAP50-95','D0DIRECT':control['metrics']['bottom3_class_map50_95'],'AF2DIRECT':candidate['metrics']['bottom3_class_map50_95'],'delta':delta['bottom3_class_map50_95']},
    {'metric':'Worst-class mAP50-95','D0DIRECT':control['metrics']['worst_class_map50_95'],'AF2DIRECT':candidate['metrics']['worst_class_map50_95'],'delta':delta['worst_class_map50_95']},
    {'metric':'Raw top-500 proposal accessibility','D0DIRECT':control['diagnostic']['raw_top500_proposal_accessibility'],'AF2DIRECT':candidate['diagnostic']['raw_top500_proposal_accessibility'],'delta':delta['raw_top500_proposal_accessibility']},
    {'metric':'Localization-conditioned Top-1','D0DIRECT':control['diagnostic']['localization_conditioned_top1'],'AF2DIRECT':candidate['diagnostic']['localization_conditioned_top1'],'delta':delta['localization_conditioned_top1']},
    {'metric':'Correct-decision recall','D0DIRECT':control['diagnostic']['correct_decision_recall'],'AF2DIRECT':candidate['diagnostic']['correct_decision_recall'],'delta':delta['correct_decision_recall']},
]
display(pd.DataFrame(rows).style.format({'D0DIRECT':'{:.2%}','AF2DIRECT':'{:.2%}','delta':'{:+.2%}'}))

print('SCREEN:',json.dumps(summary['screen'],indent=2))
print('EPOCHS:',{'D0DIRECT':control['completed_epochs'],'AF2DIRECT':candidate['completed_epochs']})
print('PRETRAINED SHA:',static['pretrained_checkpoint_sha256'])
print('COMMON INITIAL DETECTOR SHA:',static['common_initialized_detector_state_sha256'])
print('TEST:',summary['test_images_accessed'])
print('STATE ZIP:',STATE_ZIP)
print('Kirim tabel ini + SCREEN. Jangan buka test.')
